# DINOv2 — last-block fine-tuning

The actual DINOv2 ViT-S/14 architecture is in the local `dinov2_model.py` file. Only Transformer block 12 and the Low/High head are updated.


In [1]:
import sys
import torch
from pathlib import Path

MODEL_FOLDER = (
    Path.home() / "Desktop/Paper replication/model_reproductions_7_models"
    / "05_dinov2_fine_tuned"
)
sys.path.insert(0, str(MODEL_FOLDER))

from dinov2_model import build_dinov2_small
from vit_training_helpers import configure_parameters, set_seed, train_fine_tuned_stable


In [2]:
set_seed(42)
model = build_dinov2_small()
initial_checkpoint = (
    MODEL_FOLDER.parent / '04_dinov2_frozen/notebook_results/best_validation_accuracy.pth'
)
model.load_state_dict(torch.load(initial_checkpoint, map_location='cpu', weights_only=True))
configure_parameters(model, phase='fine_tuned', last_stage=model.blocks[-1])
print(model.blocks[-1])
print('Trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad))


TransformerBlock(
  (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
  (attn): Attention(
    (qkv): Linear(in_features=384, out_features=1152, bias=True)
    (proj): Linear(in_features=384, out_features=384, bias=True)
  )
  (ls1): LayerScale()
  (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
  (mlp): Mlp(
    (fc1): Linear(in_features=384, out_features=1536, bias=True)
    (act): GELU(approximate='none')
    (fc2): Linear(in_features=1536, out_features=384, bias=True)
  )
  (ls2): LayerScale()
)
Trainable: 1776002


In [3]:
train_fine_tuned_stable(
    model=model, last_stage=model.blocks[-1],
    output_dir=MODEL_FOLDER/'notebook_results_cpu_notebook',
    epochs=10, learning_rate=1e-5, batch_size=8, center_crop=True,
)


{"epoch": 1, "train_loss": 0.3480639396743341, "train_accuracy": 0.875, "validation_loss": 0.26712050173017715, "validation_accuracy": 0.9555555555555556}


{"epoch": 2, "train_loss": 0.25724759909578343, "train_accuracy": 0.9034090909090909, "validation_loss": 0.2393811994128757, "validation_accuracy": 0.8666666666666667}


{"epoch": 3, "train_loss": 0.29746525937860663, "train_accuracy": 0.9147727272727273, "validation_loss": 0.2255883942047755, "validation_accuracy": 0.9111111111111111}


{"epoch": 4, "train_loss": 0.2154042603532699, "train_accuracy": 0.9090909090909091, "validation_loss": 0.2076404266887241, "validation_accuracy": 0.9111111111111111}


{"epoch": 5, "train_loss": 0.22314748893999917, "train_accuracy": 0.9147727272727273, "validation_loss": 0.20304247389237087, "validation_accuracy": 0.9333333333333333}


...

{"epoch": 7, "train_loss": 0.27486713327446277, "train_accuracy": 0.9318181818181818, "validation_loss": 0.16735916700628067, "validation_accuracy": 0.9333333333333333}


{"epoch": 8, "train_loss": 0.20333511477061125, "train_accuracy": 0.9431818181818182, "validation_loss": 0.26719138423601785, "validation_accuracy": 0.9555555555555556}


{"epoch": 9, "train_loss": 0.2021093158499835, "train_accuracy": 0.9318181818181818, "validation_loss": 0.1444675621887048, "validation_accuracy": 0.9555555555555556}


{"epoch": 10, "train_loss": 0.262499960337829, "train_accuracy": 0.9545454545454546, "validation_loss": 0.2116032067272398, "validation_accuracy": 0.8888888888888888}


{
  "final": {
    "validation_loss": 0.2116032067272398,
    "validation_accuracy": 0.8888888888888888,
    "correct_predictions": 40,
    "incorrect_predictions": 5,
    "prediction_count": 45,
    "confusion_matrix": [
      [
        23,
        1
      ],
      [
        4,
        17
      ]
    ],
    "Low": {
      "precision": 0.8518518518518519,
      "recall": 0.9583333333333334,
      "f1-score": 0.9019607843137255,
      "support": 24.0
    },
    "High": {
      "precision": 0.9444444444444444,
      "recall": 0.8095238095238095,
      "f1-score": 0.8717948717948718,
      "support": 21.0
    },
    "macro_f1": 0.8868778280542986,
    "weighted_f1": 0.8878833584715937
  },
  "best_loss": {
    "validation_loss": 0.1444675621887048,
    "validation_accuracy": 0.9555555555555556,
    "correct_predictions": 43,
    "incorrect_predictions": 2,
    "prediction_count": 45,
    "confusion_matrix": [
      [
        23,
        1
      ],
      [
        1,
        20
      ]
   